# IPL Intelligence — 02. Batting Performance & Player Profiles

Build reproducible player summaries: runs, balls faced, strike rate, average, boundary dependence, dot-ball rate, phase performance and season consistency. Minimum sample thresholds are essential for fair comparisons.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "deliveries.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
df = pd.read_csv(DATA / "deliveries.csv", low_memory=False)
matches = pd.read_csv(DATA / "matches.csv", low_memory=False)
for c in ["start_date"]:
    if c in df: df[c] = pd.to_datetime(df[c], errors="coerce")
for c in ["runs_off_bat","extras","total_runs","legal_ball","is_wicket","is_bowler_credited_wicket","over_number"]:
    if c in df: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
df["season"] = df["season"].astype(str)
print(f"Deliveries: {len(df):,} | Matches: {df.match_id.nunique():,} | Seasons: {df.season.nunique()}")

## 1. Construct batting metrics

In [ ]:
# Legal balls faced excludes wides; no-balls count as balls faced in standard batting scorecards.
# Cricsheet's is_legal_ball excludes wides and no-balls, so derive batter balls faced separately.
wide = pd.to_numeric(df.get("wides", 0), errors="coerce").fillna(0)
df["batter_ball_faced"] = (wide == 0).astype(int)
df["four"] = df.runs_off_bat.eq(4).astype(int)
df["six"] = df.runs_off_bat.eq(6).astype(int)
bat = df.groupby("striker", as_index=False).agg(
    runs=("runs_off_bat","sum"), balls=("batter_ball_faced","sum"),
    fours=("four","sum"), sixes=("six","sum"),
    dismissals=("player_dismissed", lambda s: s.notna().sum()),
    innings=("innings_key","nunique"), matches=("match_id","nunique"),
    dot_deliveries=("runs_off_bat", lambda s: s.eq(0).sum())
).rename(columns={"striker":"player"})
bat["strike_rate"] = np.where(bat.balls>0, 100*bat.runs/bat.balls, np.nan)
bat["average"] = np.where(bat.dismissals>0, bat.runs/bat.dismissals, np.nan)
bat["boundary_runs_pct"] = np.where(bat.runs>0, 100*(4*bat.fours+6*bat.sixes)/bat.runs, np.nan)
bat["dot_ball_pct"] = np.where(bat.balls>0, 100*bat.dot_deliveries/bat.balls, np.nan)
display(bat.sort_values("runs", ascending=False).head(20).round(2))

## 2. Leaderboards with minimum opportunity

In [ ]:
MIN_BALLS = 500  # Change to 120/300/500+ depending on the question
qualified = bat[bat.balls >= MIN_BALLS].copy()
print(f"Qualified batters (at least {MIN_BALLS} balls): {len(qualified)}")
print("\nTop run scorers"); display(qualified.nlargest(15,"runs")[["player","matches","innings","runs","balls","average","strike_rate"]].round(2))
print("\nHighest strike rates among qualified batters"); display(qualified.nlargest(15,"strike_rate")[["player","matches","runs","balls","average","strike_rate","dot_ball_pct"]].round(2))

## 3. Volume vs efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(9,6)); ax.scatter(qualified.strike_rate, qualified.average, s=np.maximum(qualified.runs/20, 12), alpha=.65); ax.set_xlabel("Strike rate"); ax.set_ylabel("Batting average"); ax.set_title("Batting average vs strike rate (bubble size ~ runs)"); fig.tight_layout()

## 4. A player profile over phases

In [ ]:
player_name = "RG Sharma"  # Change to any exact striker label in the dataset
p = df[df.striker.eq(player_name)].copy()
if p.empty:
    print("No exact label found. Similar names:", [x for x in sorted(df.striker.dropna().unique()) if player_name.lower() in x.lower()][:20])
else:
    pwide = pd.to_numeric(p.wides, errors="coerce").fillna(0)
    p["batter_ball_faced"] = (pwide == 0).astype(int)
    prof = p.groupby("phase", as_index=False).agg(runs=("runs_off_bat","sum"), balls=("batter_ball_faced","sum"), dismissals=("player_dismissed",lambda s:s.notna().sum()), fours=("runs_off_bat",lambda s:s.eq(4).sum()), sixes=("runs_off_bat",lambda s:s.eq(6).sum()))
    prof["strike_rate"] = np.where(prof.balls>0,100*prof.runs/prof.balls,np.nan)
    display(prof)
    fig, ax = plt.subplots(figsize=(8,4)); ax.bar(prof.phase.astype(str), prof.runs); ax.set_title(f"{player_name}: runs by innings phase"); ax.set_ylabel("Runs"); fig.tight_layout()

## 5. Season consistency

In [ ]:
player_season = df.groupby(["season","striker"], as_index=False).agg(runs=("runs_off_bat","sum"), balls=("match_id","size"), matches=("match_id","nunique"))
player_season = player_season[player_season.striker.eq("RG Sharma")].sort_values("season")
display(player_season)
if not player_season.empty:
    fig, ax = plt.subplots(figsize=(11,4)); ax.plot(player_season.season, player_season.runs, marker="o"); ax.set_title("Rohit Sharma: recorded runs by season"); ax.tick_params(axis="x",rotation=60); fig.tight_layout()

## Important interpretation limits
- This data does not include shot type, so it cannot validate a claim such as “strong on pull shots but often dismissed playing the pull.”
- A zero-run delivery is not always a batter-attributable dot in every scorecard convention; label the chosen definition clearly.
- Dismissal counts based on `player_dismissed` can include run-outs and other dismissals not credited to the bowler.
- Compare players within similar roles and eras where possible.